# Vender mais ou ganhar mais?

Dados integralmente simulados. Análise de qualidade, métricas, retenção e um experimento independente do histórico.


In [1]:
from pathlib import Path
import json
import pandas as pd
from retail_lab.pipeline import run
from retail_lab.warehouse import query

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "sql" / "warehouse.sql").exists()
result = run(ROOT, seed=42)
DATABASE = ROOT / "data" / "warehouse.sqlite"
print("Snapshot simulado reproduzido com seed 42.")


Snapshot simulado reproduzido com seed 42.


## Qualidade antes da análise

Cópias idênticas são contadas e deduplicadas. Chaves conflitantes ou referências inválidas interrompem o processamento.


In [2]:
quality = json.loads((ROOT / "reports" / "data_quality.json").read_text())
print(pd.DataFrame(quality).T.to_string())


             input_rows  accepted_rows  exact_duplicates_removed
customers          6000           6000                         0
products             24             24                         0
orders            24020          24000                        20
order_items       47866          47866                         0


## Volume e resultado

Contribuição é receita após descontos menos CMV e logística. Não equivale a lucro líquido. A margem é a razão das somas, não a média das margens.


In [3]:
monthly = query(DATABASE, "SELECT * FROM mart_monthly ORDER BY month")
print(monthly.round(3).to_string(index=False))
print("\nCategorias — margem bruta sem logística:")
print(query(DATABASE, "SELECT * FROM mart_category ORDER BY gross_margin DESC").round(3).to_string(index=False))


  month  delivered_orders  active_customers  net_revenue  contribution  average_ticket  contribution_margin
2025-01              1679              1458   1301298.25     440882.52          775.04                0.339
2025-02              1507              1331   1143231.48     391291.58          758.61                0.342
2025-03              1704              1465   1310206.54     443267.95          768.90                0.338
2025-04              1670              1459   1275249.50     431174.31          763.62                0.338
2025-05              1705              1483   1281331.33     434408.46          751.51                0.339
2025-06              1574              1390   1201276.46     406737.22          763.20                0.339
2025-07              1682              1474   1268676.45     430636.65          754.27                0.339
2025-08              1619              1421   1225411.32     414497.58          756.89                0.338
2025-09              1666   

## Retenção e segmentos

Coorte é a primeira compra observada em 2025. Uma janela ainda não observada fica ausente, não zero. RFM é heurístico e não prevê churn.


In [4]:
cohort = pd.read_csv(ROOT / "reports" / "cohorts.csv")
print(cohort.pivot(index="cohort", columns="age_month", values="retention").tail(6).round(3).to_string())
print("\nSegmentos RFM:")
print(pd.read_csv(ROOT / "reports" / "rfm_summary.csv").to_string(index=False))


age_month    0      1      2      3
cohort                             
2025-07    1.0  0.231  0.253  0.214
2025-08    1.0  0.214  0.266  0.370
2025-09    1.0  0.269  0.391  0.353
2025-10    1.0  0.421  0.286    NaN
2025-11    1.0  0.351    NaN    NaN
2025-12    1.0    NaN    NaN    NaN

Segmentos RFM:
                       segment  customers  average_recency  average_frequency    revenue
               Demais clientes       2344            55.55               2.93 5076927.79
           Recentes frequentes       2258            27.65               5.39 9036155.42
Sem compra ha mais de 120 dias       1225           201.49               2.47 2310974.45


## Experimento

Todos os sorteados entram na métrica primária de contribuição por cliente. Conversão é secundária. Sorteio 50/50 esperado; horizonte fixo simulado de 14 dias; Welch bilateral e bootstrap por cliente.


In [5]:
print(pd.DataFrame(result["groups"]).T.round(4).to_string())
print("\nDiferença de contribuição:", round(result["difference"], 2))
print("IC 95% Welch:", [round(v, 2) for v in result["ci_95"]])
print("IC 95% bootstrap:", [round(v, 2) for v in result["bootstrap_ci_95"]])
print("SRM p:", round(result["srm_p_value"], 4))
print("Decisão:", result["decision"])


               n  conversion  mean_contribution
control   7888.0      0.2036            13.6071
discount  8112.0      0.2681             7.9970

Diferença de contribuição: -5.61
IC 95% Welch: [-6.36, -4.86]
IC 95% bootstrap: [-6.37, -4.9]
SRM p: 0.0766
Decisão: Nao expandir o desconto


## Checagem independente do denominador

Selecionar apenas compradores após o tratamento mudaria o estimando. Aqui, dividimos a contribuição pela quantidade de todos os sorteados.


In [6]:
experiment = pd.read_csv(ROOT / "data" / "raw" / "experiment.csv")
check = experiment.groupby("variant").agg(randomized=("customer_id", "count"), buyers=("converted", "sum"), contribution=("contribution", "sum"))
check["per_randomized_customer"] = check.contribution / check.randomized
print(check.round(3).to_string())


          randomized  buyers  contribution  per_randomized_customer
variant                                                            
control         7888    1606     107332.87                   13.607
discount        8112    2175      64871.69                    7.997


## Conclusão

No cenário padrão, a conversão aumenta e a contribuição cai. O efeito foi programado no gerador: demonstra o método, não um resultado comercial real. A série histórica não permite atribuir causalmente o pico de novembro ao desconto. Incentivos menores ou segmentados exigiriam outro experimento.

[Desenho experimental](../docs/experiment_design.md) · [Relatório executivo](../reports/executive_summary.md)
